# `@lru_cache` in Python - From Basics to Expert

## What is it?
- `@lru_cache` is a decorator from Python's `functools` module that **memoizes** a function - it remembers the result of previous cals and returns the cached result when the same inputs are seen again, instead of recomputing.
- LRU stands for Least Recently used - when the cache fills up, it evicts the result that accessed least recently to make room for new ones

In [ ]:
from functools import lru_cache

## Why use it?
- speed - avoid repeating expensive computations
- simplicity - one decorator line - no manual dict caching
- correctness - only works on pure functions (same input - same output, no side effects), which is a healthy constraint

The core idea is trading memory for speed.

## When to use it?
#### Good candidates:
- recursive funcitons with overlapping subproblems (Fibonacci, tree traversal)
- pure functions called repeatedly with the same arguments
- expnsive computations: parsing, math, string processing
- database style ookups where data doesn't change

#### Bad candidates:
- functions with side effects (writing to files, printing, mutating state)
- funcitons where the output changes over time (current time, random numbers, live API calls)
- functions with unhashable arguments (lists, dicts) - these will raise a `TypeError`

## The Baseline: Without Cache
Before seeing the cache, understand the problem it solves

In [ ]:
import time

def slow_square(n):
    time.sleep(0.5) # simulate expensive work
    return n * n

# called 4 times, pays the cost every time
start = time.time()
results = [slow_square(5), slow_square(5), slow_square(5), slow_square(5)]

print(f"Took {time.time() - start:.2f}") #~ 2s
print(results)

# Same input, same output, computed four times - WASTEFUL

## Basic usage

In [ ]:
@lru_cache(maxsize=128)
def slow_square(n):
    time.sleep(0.5)
    return n * n

start = time.time()
results = [slow_square(5), slow_square(5), slow_square(5), slow_square(5)]

print(f"Took {time.time() - start:.2f}") #~ 0.5s - only computed once
print(results)

- the efirtst call computes ans stores the result - the next three calls return instantly from cache
- `maxsize` controls hoe many unique results to store - when the cache is full - the east recently used enrty is dropped

## The Classic example - Fibonacci
- Fubonacci is the poster child for `lru_cache` because naive recursion recomputes the same values exponentially

In [ ]:
# WITHOUT CACHE - exponential time O(2^n)

def fib_slow(n):
    if n < 2:
        return n
    return fib_slow(n - 1) + fib_slow(n - 2)

@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

start = time.time()
print(fib_slow(25))
print(f"Without cache: {time.time() - start:.3f}s")

start = time.time()
print(fib(25))
print(f"with cache: {time.time() - start:.6f}s")

- `maxsize=None` means unlimited cache - useful when you know the input space is bounded (like `fib(n)` for a fixed max n)

## Inspecting the Cache
- `lru_cache` gives you `.cache_info()` method to peek inside

In [ ]:
@lru_cache(maxsize=4)
def compute(n):
    return n ** 3

# warm up the cache
for i in range(6):
    compute(i)

print(compute.cache_info())


In [ ]:
# already cached
compute(5)
compute(5)

print(compute.cache_info())

- hits - how many times a cached results was returned
- misses - how many times the funciton actually ran
- maxsize - the limit you set
- currsize - how many entries are currently stored

- A high hit/mixx ration means the cache is working well
- a low ration means you are caching things that are rearly reused - not worth the memeory

## Cleating the Cache

In [ ]:
@lru_cache(maxsize=128)
def get_data(key):
    print(f" computing for {key}...")
    return key.upper()

get_data("hello") # computing for "hello"
get_data("hello") # from cache - no print
get_data("world") # computing for world

print(get_data.cache_info())

In [ ]:
get_data.cache_clear()
print(get_data.cache_info())
get_data("hello") # computing for hello... cache was cleared

# use `.cache_clear()` when underlying data changes and you need fresh results

## `maxsize=None` VS `maxsize=128` VS `maxsize=0`

In [ ]:
@lru_cache(maxsize=None) #unlimited cache - frows forever
def func1(n): return n * 2

@lru_cache(maxsize=128) # fixed cache - LRU evitction when full (default)
def func2(n): return n * 2

@lru_cache(maxsize=0) # cache disabled entirely - useful for testing or toggling
def func3(n): return n * 2

# shorthand for maxsize=128 (python 3.8+)
@lru_cache
def func4(n): return n * 2

### Rule of thumb
- `maxsize=None` when inputs are bounded (recursion, small fixed domains)
- `maxsize=128` (or custom) for oprn-ended inputs - cap memeory usage
- `maxsize=0` to disable caching (useful in tests to isolate behavior)

## Arguments must be hashable
- the cache works by using the function arguments as dictionary keys - this means arguments must be hashable

In [ ]:
@lru_cache(maxsize=128)
def process(data):
    print(f" data tyoe {type(data)}")
    return sum(data)

In [ ]:
# works fine - tuples are hashable
print(process((1,2,3))) # 6
print(process((1,2,3))) # 6 from cache

In [ ]:
# raises TypeError - lists are NOThashable
try:
    process([1 ,2, 3])
except TypeError as e:
    print(f"TypeError: {e}")

In [ ]:
# THE FIX: convert mutable types to immutable ones before passing in

@lru_cache(maxsize=128)
def process(data_tuple):
    return sum(data_tuple)

my_list = [1, 2, 3]
result = process(tuple(my_list)) # convert at the call site
print(result)